In [1]:
import warnings
warnings.simplefilter("ignore", FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import time

# LightGBM
from lightgbm import LGBMClassifier,early_stopping, log_evaluation


# sklearn

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.experimental import enable_halving_search_cv 

from sklearn.model_selection import (
    train_test_split, KFold, cross_validate, cross_val_score,
    RandomizedSearchCV, StratifiedKFold,train_test_split,HalvingRandomSearchCV
)

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score,
                             average_precision_score,roc_curve)


from sklearn.base import BaseEstimator, TransformerMixin, clone
from scipy.stats import randint, uniform, loguniform

# Importações locais
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.preprocess_utils_diab3a import * #(NOVO atualizações)
from src.plot_metrica_class import *

print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))
start_inicial = time.time()


#Processo iniciado em: 11:31:07


In [2]:
BASE = Path.cwd().parent   
# =====================================================
# ⚙️ 0. carregamento dos preprocessador 
# =====================================================
PP3a = joblib.load(BASE/'src'/'preprocess_diabetes_v3a.joblib')['preprocessador']



# # =====================================================
# # 📁 1. Leitura dos dados & Separação das bases
# # =====================================================
DATA_DIR = BASE / "data" / "raw"
TARGET="diagnosed_diabetes"
df = pd.read_csv(DATA_DIR / "train.csv").reset_index(drop=True)
X_train=df.drop(columns=TARGET)
y_train=df[TARGET]


# =====================================================
# 📁 2. Modelo e pipeline
# =====================================================
DATA_MODELS= BASE /"models"
#pipe_LGBM1=joblib.load(DATA_MODELS / 'modelo_LGBM2_final_randsearch_e.roc_auc_v3a.joblib')
pipe_LGBM3=joblib.load(DATA_MODELS / 'modelo_LGBM2_final_Hrandsearch_e.roc_auc_v3a.joblib')


In [3]:
print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))
# # =====================================================
# # 3.Treino & Teste
# # =====================================================
pipe_LGBM3.fit(X_train, y_train)



#Processo iniciado em: 11:31:08


Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('ordinal_map', OrdinalMapper()),
                                 ('feature_engineering',
                                  RiskFeatureEngineer2()),
                                 ('preprocess',
                                  ColumnTransformer(transformers=[('num',
                                                                   Pipeline(steps=[('imp',
                                                                                    SimpleImputer(strategy='median')),
                                                                                   ('float32',
                                                                                    FunctionTransformer(func=<function to_float32 at 0x7bb95a334d30>))]),
                                                                   Index(['age', 'alcohol_consumption_per_...
                                                                    'cardiovascular_history'])],
                                                    verbose_feature_names_out=False))])),
                ('model',
                 LGBMClassifier(colsample_bytree=0.15180288401497988,
                                force_row_wise=True,
                                learning_rate=0.047436809713707416, max_depth=7,
                                min_child_samples=44, n_estimators=1338,
                                n_jobs=-1, num_leaves=114, objective='binary',
                                random_state=42, reg_alpha=0.5853350821380293,
                                reg_lambda=3, subsample=0.8752120040700155,
                                verbose=-1))])

In [4]:
print("\n#Processo iniciado em:", time.strftime("%H:%M:%S"))

# =====================================================
# Submissão Kaggle
# =====================================================
DATA_DIR = BASE / "data" / "raw"
base = pd.read_csv(DATA_DIR / "test.csv")

id_test = base["id"]

x_test=base.drop(columns='id')

# Previsão
y_pred=pipe_LGBM3.predict(x_test)

#  DataFrame de submissão
submission = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred
})
submission_path = ("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/process/submission_LGBM2_final_Hrandsearch_e.roc_auc_v3a.csv")
submission.to_csv(submission_path, index=False)
print(f"✅ Arquivo de submissão salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")


#Processo iniciado em: 11:31:41
✅ Arquivo de submissão salvo • 12/03/2026-11:31:52
